# Allenamento del modello loghi su Google Colab

Colab offre una GPU gratuita: l'allenamento è molto più veloce che su CPU.
Menu **Runtime > Cambia tipo di runtime > GPU**, poi esegui le celle in ordine.

Prima di caricare immagini su un servizio esterno controlla che non contengano dati personali.

In [ ]:
# Cambia l'indirizzo con quello del tuo repository
REPO_URL = "https://github.com/TUO_UTENTE/photo-cleanser.git"

!git clone $REPO_URL photo-cleanser
%cd photo-cleanser
!pip -q install ultralytics

## 1. Carica il dataset

Crea uno ZIP con le immagini e le etichette YOLO (un `.txt` per immagine, righe `classe cx cy w h`).
Per annotazioni Pascal VOC (XML) usa prima `training/voc_to_yolo.py`.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
!rm -rf training/source && mkdir -p training/source
!unzip -q "$zip_name" -d training/source

In [ ]:
# Solo se hai annotazioni XML (OpenLogo, LogoDet-3K): le converte in etichette YOLO
# !python training/voc_to_yolo.py --source training/source --dest training/converted
# SOURCE = "training/converted"
SOURCE = "training/source"

!python training/prepare_dataset.py --source $SOURCE --dest training/dataset --force

## 2. Allena

In [ ]:
!python training/train_logo.py --data training/dataset/data.yaml --epochs 80 --device 0 --no-install

## 3. Scarica il modello

Il file migliore è `training/runs/logo/weights/best.pt`. Scaricalo, rinominalo `logo.pt` e mettilo nella cartella `models/` del progetto (sul tuo computer o in Codespaces).
Poi scrivi lo SHA-256 stampato qui sotto nella voce `logo` di `models/models.lock.json`, oppure avvia il sito con `ALLOW_UNVERIFIED=1`.


In [ ]:
import hashlib
from google.colab import files

best = "training/runs/logo/weights/best.pt"
print("sha256:", hashlib.sha256(open(best, "rb").read()).hexdigest())
files.download(best)